In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# CHỌN thư mục lưu (thay đổi nếu muốn)
DUMP_ROOT = "/content/drive/MyDrive/crypto_data"   # lưu trên Drive (khuyên dùng)
# DUMP_ROOT = "/content"                            # hoặc lưu tạm trên máy ảo Colab

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip list


Package                               Version
------------------------------------- -------------------
absl-py                               1.4.0
absolufy-imports                      0.3.1
accelerate                            1.10.1
aiofiles                              24.1.0
aiohappyeyeballs                      2.6.1
aiohttp                               3.13.0
aiosignal                             1.4.0
alabaster                             1.0.0
albucore                              0.0.24
albumentations                        2.0.8
ale-py                                0.11.2
alembic                               1.16.5
altair                                5.5.0
annotated-types                       0.7.0
antlr4-python3-runtime                4.9.3
anyio                                 4.11.0
anywidget                             0.9.18
argon2-cffi                           25.1.0
argon2-cffi-bindings                  25.1.0
array_record                          0.8.1
arrow 

In [ ]:
import os
import warnings
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta, date, time
from binance_historical_data import BinanceDataDumper

warnings.filterwarnings('ignore')

# =========== CONFIG ===========
ASSET_CLASS = "um"          # USDT-M Futures
DATA_FREQUENCY = "1d"       # 1d cho 5 năm để nhẹ + đủ backtest
TODAY = datetime.utcnow().date()
START_DATE = (TODAY - timedelta(days=5*365)).strftime("%Y-%m-%d")
END_DATE   = TODAY.strftime("%Y-%m-%d")

# Bỏ stable/staked/wrapped: chỉ còn 7 mã perp phổ biến
SYMBOLS = ["BTCUSDT","ETHUSDT","BNBUSDT","XRPUSDT","SOLUSDT","TRXUSDT","DOGEUSDT"]

# Thư mục gốc dump (từ Cell 0). Nếu bạn không chạy Cell 0, set DUMP_ROOT = "/content"
try:
    DUMP_ROOT
except NameError:
    DUMP_ROOT = "/content"

os.makedirs(DUMP_ROOT, exist_ok=True)
print(f"Dump root: {DUMP_ROOT}")
print(f"Range: {START_DATE} → {END_DATE} | Asset: {ASSET_CLASS} | Freq: {DATA_FREQUENCY}")
print("Symbols:", SYMBOLS)

Dump root: /content/drive/MyDrive/crypto_data
Range: 2020-10-17 → 2025-10-16 | Asset: um | Freq: 1d
Symbols: ['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'XRPUSDT', 'SOLUSDT', 'TRXUSDT', 'DOGEUSDT']


In [ ]:
import time, io, zipfile, requests
from urllib.error import URLError
from calendar import monthrange

# ========= Các hàm giữ nguyên phong cách cũ =========

def get_list_all_trading_pairs(asset_class=ASSET_CLASS, data_frequency=DATA_FREQUENCY):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=DUMP_ROOT,
        asset_class=asset_class,        # "um" = USDT-M futures
        data_type="klines",
        data_frequency=data_frequency,
    )
    return data_dumper.get_list_all_trading_pairs()

def filter_usdt_tickers(tickers):
    exclude_keywords = ["UPUSDT", "DOWNUSDT", "BEARUSDT", "BULLUSDT"]
    return [t for t in tickers if t.endswith("USDT") and not any(ex in t for ex in exclude_keywords)]

def find_first_data_date(ticker, asset_class=ASSET_CLASS, data_frequency=DATA_FREQUENCY):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=DUMP_ROOT,
        asset_class=asset_class,
        data_type="klines",
        data_frequency=data_frequency,
    )
    return data_dumper.get_min_start_date_for_ticker(ticker)

def detect_timestamp_unit(timestamp):
    num_digits = len(str(int(timestamp)))
    if num_digits == 13: return 'ms'
    if num_digits == 16: return 'us'
    return 'ms'  # fallback

def convert_timestamp(ts):
    return pd.to_datetime(ts, unit=detect_timestamp_unit(ts), errors='coerce')

def read_csv_file(file_path):
    df = pd.read_csv(file_path)
    df.columns = [
        "open_time", "Open", "High", "Low", "Close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    df['open_time']  = df['open_time'].apply(convert_timestamp)
    df['close_time'] = df['close_time'].apply(convert_timestamp)
    return df

def get_csv_files(directory):
    try:
        if not os.path.exists(directory):
            return []
        return [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.csv')]
    except Exception as e:
        print(f"Lỗi khi đọc thư mục {directory}: {str(e)}")
        return []

# ========= Fallback tầng 1: REST với nhiều endpoint =========
FAPI_ENDPOINTS = [
    "https://fapi.binance.com",
    "https://fapi1.binance.com",
    "https://fapi.binancefuture.com"
]

def _to_millis(dt_str_YYYY_MM_DD):
    dt = datetime.strptime(dt_str_YYYY_MM_DD, "%Y-%m-%d")
    # dùng time.mktime để có epoch (UTC colab ổn)
    return int(time.mktime(dt.timetuple()) * 1000)

def fetch_klines_rest_multi(symbol, interval="1d", start_date=START_DATE, end_date=END_DATE, limit=1500, sleep_sec=0.2):
    """Thử lần lượt nhiều endpoint fapi; nếu endpoint bị 451 hoặc DNS fail sẽ thử cái tiếp theo."""
    start_ms = _to_millis(start_date)
    end_ms   = _to_millis(end_date)
    out = []
    cur = start_ms

    for base in FAPI_ENDPOINTS:
        try:
            tmp = []
            cur_ptr = cur
            while True:
                params = {
                    "symbol": symbol,
                    "interval": interval,
                    "startTime": cur_ptr,
                    "endTime": end_ms,
                    "limit": limit
                }
                r = requests.get(f"{base}/fapi/v1/klines", params=params, timeout=20)
                if r.status_code == 451:
                    raise requests.HTTPError(f"451 from {base}")
                r.raise_for_status()
                data = r.json()
                if not data:
                    break
                tmp.extend(data)
                last_open = data[-1][0]
                next_ms = last_open + 1
                if next_ms >= end_ms:
                    break
                cur_ptr = next_ms
                time.sleep(sleep_sec)
            if tmp:
                out = tmp
                break  # thành công với base này
        except Exception as e:
            print(f"REST trên {base} lỗi: {type(e).__name__}: {e}")
            continue

    if not out:
        return None

    cols = [
        "open_time","Open","High","Low","Close","volume",
        "close_time","quote_asset_volume","number_of_trades",
        "taker_buy_base_asset_volume","taker_buy_quote_asset_volume","ignore"
    ]
    df = pd.DataFrame(out, columns=cols)
    return df

# ========= Fallback tầng 2: tải ZIP từ data.binance.vision (monthly) =========
VISION_BASE = "https://data.binance.vision"

def month_iter(start_date_str, end_date_str):
    """Sinh (yyyy, mm) từ START_DATE → END_DATE (bao gồm)."""
    s = datetime.strptime(start_date_str, "%Y-%m-%d").date()
    e = datetime.strptime(end_date_str,   "%Y-%m-%d").date()
    y, m = s.year, s.month
    while (y < e.year) or (y == e.year and m <= e.month):
        yield y, m
        if m == 12:
            y += 1; m = 1
        else:
            m += 1

def download_month_zip(symbol, interval="1d", yyyy=2020, mm=1, asset_class=ASSET_CLASS, dump_root=DUMP_ROOT):
    """
    Tải 1 tháng dữ liệu futures UM klines dạng ZIP từ data.binance.vision:
    /data/futures/um/monthly/klines/<SYMBOL>/<interval>/<SYMBOL>-<interval>-YYYY-MM.zip
    Giải nén và lưu CSV vào um/daily/klines/<SYMBOL>/<interval>/...
    """
    file_name = f"{symbol}-{interval}-{yyyy}-{mm:02d}.zip"
    url = f"{VISION_BASE}/data/futures/{asset_class}/monthly/klines/{symbol}/{interval}/{file_name}"

    r = requests.get(url, timeout=30)
    if r.status_code == 404:
        return False  # tháng này chưa có data
    r.raise_for_status()

    # Giải nén từ bytes
    z = zipfile.ZipFile(io.BytesIO(r.content))
    out_dir = os.path.join(dump_root, f"{asset_class}/daily/klines/{symbol}/{interval}")
    os.makedirs(out_dir, exist_ok=True)
    z.extractall(out_dir)
    return True

def download_vision_monthly_range(symbol, start_date=START_DATE, end_date=END_DATE, interval="1d"):
    """Cố tải theo từng tháng; trả về True nếu có ít nhất 1 tháng tải được."""
    ok_any = False
    for y, m in month_iter(start_date, end_date):
        try:
            ok = download_month_zip(symbol, interval=interval, yyyy=y, mm=m)
            if ok:
                ok_any = True
                print(f"  ✅ vision: {symbol} {y}-{m:02d}")
            else:
                print(f"  ⛔ vision: {symbol} {y}-{m:02d} (404)")
        except Exception as e:
            print(f"  ⚠️ vision lỗi {symbol} {y}-{m:02d}: {type(e).__name__}: {e}")
    return ok_any

# ========= Hàm download chính: giữ API cũ + 3 lớp fallback =========
def download_ticker(ticker, date_start, date_end, data_frequency=DATA_FREQUENCY, asset_class=ASSET_CLASS):
    """
    1) Thử BinanceDataDumper (sử dụng data.binance.vision chuẩn)
    2) Nếu lỗi URLError/DNS/451 → thử REST đa endpoint
    3) Nếu vẫn fail → tải trực tiếp monthly ZIP từ data.binance.vision
    Lưu file vào đúng thư mục um/daily/klines/<TICKER>/<FREQ>/ để process_csv_files() đọc.
    """
    # 1) Dumper
    try:
        data_dumper = BinanceDataDumper(
            path_dir_where_to_dump=DUMP_ROOT,
            asset_class=asset_class,
            data_type="klines",
            data_frequency=data_frequency,
        )
        ds = datetime.strptime(date_start, "%Y-%m-%d").date()
        de = datetime.strptime(date_end,   "%Y-%m-%d").date()
        data_dumper.dump_data(
            tickers=ticker,
            date_start=ds,
            date_end=de,
            is_to_update_existing=False,
        )
        return
    except Exception as e:
        print(f"⚠️ Dumper lỗi ({type(e).__name__}): {e}")

    # 2) REST đa endpoint
    try:
        df = fetch_klines_rest_multi(
            symbol=ticker,
            interval=data_frequency,
            start_date=date_start,
            end_date=date_end
        )
        if df is not None and not df.empty:
            out_dir = os.path.join(DUMP_ROOT, f"{asset_class}/daily/klines/{ticker}/{data_frequency}")
            os.makedirs(out_dir, exist_ok=True)
            out_csv = os.path.join(out_dir, f"REST_{date_start}_{date_end}.csv")
            df.to_csv(out_csv, index=False)
            print(f"✅ REST saved: {out_csv}")
            return
        else:
            print("REST không có dữ liệu / tiếp tục vision …")
    except Exception as e:
        print(f"⚠️ REST lỗi: {type(e).__name__}: {e} → chuyển sang vision monthly")

    # 3) Vision monthly ZIP
    ok_any = download_vision_monthly_range(
        symbol=ticker,
        start_date=date_start,
        end_date=date_end,
        interval=data_frequency
    )
    if not ok_any:
        raise RuntimeError(f"Không tải được {ticker} bằng cả Dumper, REST, và data.binance.vision.")
    else:
        print(f"✅ vision monthly đã lưu CSV cho {ticker} vào um/daily/klines/{ticker}/{data_frequency}")

def process_csv_files(ticker, asset_class=ASSET_CLASS, data_frequency=DATA_FREQUENCY):
    daily_path   = os.path.join(DUMP_ROOT, f"{asset_class}/daily/klines/{ticker}/{data_frequency}")
    monthly_path = os.path.join(DUMP_ROOT, f"{asset_class}/monthly/klines/{ticker}/{data_frequency}")

    daily_files   = get_csv_files(daily_path)
    monthly_files = get_csv_files(monthly_path)
    all_files = daily_files + monthly_files

    if not all_files:
        print(f"❗ Không có file CSV nào cho {ticker} ({asset_class}, {data_frequency})")
        return None

    data = pd.concat([read_csv_file(fp) for fp in all_files], ignore_index=True)
    data.sort_values(by='open_time', inplace=True)
    data = data.drop_duplicates(subset=['open_time'])
    return data


In [ ]:
# Crawl 7 mã (1D, 5Y) — có 3 tầng fallback
for sym in SYMBOLS:
    print(f"⬇️  Downloading {sym} ({ASSET_CLASS}, {DATA_FREQUENCY}) {START_DATE} → {END_DATE}")
    download_ticker(sym, START_DATE, END_DATE, DATA_FREQUENCY, ASSET_CLASS)

# Đọc lại & gộp
frames = {}
for sym in SYMBOLS:
    df = process_csv_files(sym, ASSET_CLASS, DATA_FREQUENCY)
    if df is None or df.empty:
        print(f"⚠️ Bỏ qua {sym} do không có dữ liệu.")
        continue
    frames[sym] = df[["open_time","Open","High","Low","Close","volume"]].rename(columns={"open_time":"date"}).set_index("date")

if not frames:
    raise RuntimeError("Không có mã nào đọc được dữ liệu. Kiểm tra lại.")

# Panel OHLCV (MultiIndex columns: (symbol, field))
panel = pd.concat(frames, axis=1).sort_index()

os.makedirs(os.path.join(DUMP_ROOT, "processed"), exist_ok=True)
panel_path = os.path.join(DUMP_ROOT, "processed/um_1d_OHLCV_5y.parquet")
panel.to_parquet(panel_path)
print("✅ Saved panel:", panel_path)

# Close-only
close_wide = pd.concat({sym: frames[sym]["Close"] for sym in frames}, axis=1)
close_wide.columns.name = "symbol"
close_csv = os.path.join(DUMP_ROOT, "processed/um_close_only_1d_5y.csv")
close_wide.to_csv(close_csv, encoding="utf-8")
print("✅ Saved close-only:", close_csv)

print("\nSố ngày dữ liệu theo mã:")
print(close_wide.notna().sum().sort_values(ascending=False))

close_wide.tail()


⬇️  Downloading BTCUSDT (um, 1d) 2020-10-17 → 2025-10-16
⚠️ Dumper lỗi (URLError): <urlopen error [Errno -2] Name or service not known>
REST trên https://fapi.binance.com lỗi: HTTPError: 451 from https://fapi.binance.com
REST trên https://fapi1.binance.com lỗi: JSONDecodeError: Expecting value: line 1 column 1 (char 0)
REST trên https://fapi.binancefuture.com lỗi: ConnectionError: HTTPSConnectionPool(host='fapi.binancefuture.com', port=443): Max retries exceeded with url: /fapi/v1/klines?symbol=BTCUSDT&interval=1d&startTime=1602892800000&endTime=1760572800000&limit=1500 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7e57b79d8d40>: Failed to resolve 'fapi.binancefuture.com' ([Errno -2] Name or service not known)"))
REST không có dữ liệu / tiếp tục vision …
  ✅ vision: BTCUSDT 2020-10
  ✅ vision: BTCUSDT 2020-11
  ✅ vision: BTCUSDT 2020-12
  ✅ vision: BTCUSDT 2021-01
  ✅ vision: BTCUSDT 2021-02
  ✅ vision: BTCUSDT 2021-03
  ✅ vision: BTCUSDT 2021-04
  ✅ v

symbol,BTCUSDT,ETHUSDT,BNBUSDT,XRPUSDT,SOLUSDT,TRXUSDT,DOGEUSDT
date,,,,,,,
2025-09-26,109588.5,4030.17,961.28,2.7823,205.01,0.33825,0.23232
2025-09-27,109577.3,4017.01,968.71,2.8032,203.32,0.33634,0.23067
2025-09-28,112119.6,4139.55,993.09,2.8662,210.70,0.33552,0.23722
2025-09-29,114257.1,4212.98,1030.43,2.8788,212.59,0.33672,0.23469
2025-09-30,113988.8,4142.75,1008.77,2.8454,208.60,0.33345,0.23282


In [ ]:
import pandas as pd
import numpy as np
import os

# Load close-only đã tạo
close_csv = os.path.join(DUMP_ROOT, "processed/um_close_only_1d_5y.csv")
close = pd.read_csv(close_csv, parse_dates=["date"]).set_index("date").sort_index()
print("Raw close shape:", close.shape)
print("Date range:", close.index.min().date(), "→", close.index.max().date())

# 1) Bỏ trùng index nếu có
close = close[~close.index.duplicated(keep="last")]

# 2) Kiểm tra lỗ hổng ngày (do crypto giao dịch 7 ngày/tuần, nên index phải là daily liên tục)
full_idx = pd.date_range(close.index.min(), close.index.max(), freq="D")
if len(full_idx) != len(close.index):
    print(f"⚠️ Phát hiện thiếu ngày: expected {len(full_idx)}, got {len(close.index)}.")
    # Gộp lại theo full index, giữ NaN để xử lý tiếp
    close = close.reindex(full_idx)

# 3) Chọn khoảng thời gian chung cho tất cả mã (từ ngày max(first_valid) đến ngày min(last_valid))
first_valid = close.apply(lambda s: s.first_valid_index())
last_valid  = close.apply(lambda s: s.last_valid_index())
start_all = max([d for d in first_valid if d is not None])
end_all   = min([d for d in last_valid  if d is not None])

aligned = close.loc[start_all:end_all].copy()
print("Aligned range:", aligned.index.min().date(), "→", aligned.index.max().date())
print("Any NaN left?", aligned.isna().any().any())

# 4) (Tuỳ chọn) xử lý NaN lẻ tẻ còn sót (không nên có sau bước 3). Nếu vẫn còn, có thể ffill nhẹ:
if aligned.isna().any().any():
    aligned = aligned.ffill().bfill()
    print("After fill, any NaN?", aligned.isna().any().any())

# 5) Tính daily log-returns & simple returns
rets = aligned.pct_change().dropna()
logrets = np.log(aligned).diff().dropna()

# 6) Lưu lại các bản đã align để backtest
out_dir = os.path.join(DUMP_ROOT, "processed")
aligned.to_parquet(os.path.join(out_dir, "um_close_only_1d_aligned.parquet"))
rets.to_parquet(os.path.join(out_dir, "um_close_returns_1d_aligned.parquet"))
logrets.to_parquet(os.path.join(out_dir, "um_close_logreturns_1d_aligned.parquet"))

print("✅ Saved aligned closes & returns into:", out_dir)
print("\nPreview aligned close tail:")
display(aligned.tail())
print("\nPreview returns tail:")
display(rets.tail())


Raw close shape: (1811, 7)
Date range: 2020-10-02 → 2025-09-30
⚠️ Phát hiện thiếu ngày: expected 1825, got 1811.
Aligned range: 2020-10-02 → 2025-09-30
Any NaN left? True
After fill, any NaN? False
✅ Saved aligned closes & returns into: /content/drive/MyDrive/crypto_data/processed

Preview aligned close tail:


,BTCUSDT,ETHUSDT,BNBUSDT,XRPUSDT,SOLUSDT,TRXUSDT,DOGEUSDT
2025-09-26,109588.5,4030.17,961.28,2.7823,205.01,0.33825,0.23232
2025-09-27,109577.3,4017.01,968.71,2.8032,203.32,0.33634,0.23067
2025-09-28,112119.6,4139.55,993.09,2.8662,210.70,0.33552,0.23722
2025-09-29,114257.1,4212.98,1030.43,2.8788,212.59,0.33672,0.23469
2025-09-30,113988.8,4142.75,1008.77,2.8454,208.60,0.33345,0.23282



Preview returns tail:


,BTCUSDT,ETHUSDT,BNBUSDT,XRPUSDT,SOLUSDT,TRXUSDT,DOGEUSDT
2025-09-26,0.006004,0.040554,0.015626,0.014697,0.064434,0.019501,0.042682
2025-09-27,-0.000102,-0.003265,0.007729,0.007512,-0.008244,-0.005647,-0.007102
2025-09-28,0.023201,0.030505,0.025167,0.022474,0.036297,-0.002438,0.028396
2025-09-29,0.019064,0.017739,0.037600,0.004396,0.008970,0.003577,-0.010665
2025-09-30,-0.002348,-0.016670,-0.021020,-0.011602,-0.018769,-0.009711,-0.007968


In [ ]:
# Chọn mã để xem
ticker = "BTCUSDT"
if ticker not in frames:
    ticker = list(frames.keys())[0]

df = frames[ticker].copy()

fig = go.Figure(
    data=[
        go.Candlestick(
            x=df.index,
            open=df["Open"],
            high=df["High"],
            low=df["Low"],
            close=df["Close"],
            name=f"{ticker} OHLC"
        )
    ]
)

fig.update_layout(
    title=f"{ticker} — OHLC (UM {DATA_FREQUENCY})",
    xaxis_title="Time",
    yaxis_title="Price",
    xaxis_rangeslider_visible=False,
    template="plotly_dark",
    height=600
)

fig.show()


In [ ]:
import os
import numpy as np
import pandas as pd

OUT_DIR = os.path.join(DUMP_ROOT, "processed")

panel_path = os.path.join(OUT_DIR, "um_1d_OHLCV_5y.parquet")
close_csv  = os.path.join(OUT_DIR, "um_close_only_1d_5y.csv")

# 0) Load lại dữ liệu đã có
panel = pd.read_parquet(panel_path)            # MultiIndex columns: (symbol, field)
close = pd.read_csv(close_csv, parse_dates=["date"]).set_index("date").sort_index()

print("Loaded panel:", panel.shape, "| close:", close.shape)
print("Date range (raw close):", close.index.min().date(), "→", close.index.max().date())

# 1) Loại trùng index, sort bảo đảm
close = close[~close.index.duplicated(keep="last")].sort_index()
panel = panel[~panel.index.duplicated(keep="last")].sort_index()

# 2) CLEAN OHLCV per-symbol
symbols = sorted({c[0] for c in panel.columns})
fields  = ["Open","High","Low","Close","volume"]

def clean_one_symbol(df_ohlcv):
    """
    df_ohlcv: DataFrame 5 cột ['Open','High','Low','Close','volume'] cho 1 symbol, index = date
    Trả về df đã clean (giữ nguyên index).
    """
    df = df_ohlcv.copy()

    # a) Giá > 0, volume >= 0 (nếu không ⇢ NaN để xử lý tiếp)
    for col in ["Open","High","Low","Close"]:
        df.loc[df[col] <= 0, col] = np.nan
    df.loc[df["volume"] < 0, "volume"] = np.nan

    # b) Consistency OHLC: High ≥ max(Open, Close), Low ≤ min(Open, Close)
    max_oc = df[["Open","Close"]].max(axis=1)
    min_oc = df[["Open","Close"]].min(axis=1)
    bad_high = df["High"] < max_oc
    bad_low  = df["Low"]  > min_oc
    # Với dòng vi phạm, đánh dấu toàn bộ OHLC = NaN (giữ volume để tham khảo, hoặc cũng NaN nếu muốn)
    bad_rows = bad_high | bad_low
    if bad_rows.any():
        df.loc[bad_rows, ["Open","High","Low","Close"]] = np.nan

    # c) Nội suy/điền thiếu an toàn:
    #    - Giá: prefer forward-fill (crypto ít gap), sau đó backward-fill nếu đầu chuỗi thiếu
    #    - Volume: fill 0 cho missing lẻ tẻ (thường an toàn với daily)
    price_cols = ["Open","High","Low","Close"]
    df[price_cols] = df[price_cols].ffill().bfill()
    df["volume"]   = df["volume"].fillna(0)

    # d) Sanity check cuối: vẫn đảm bảo quan hệ High/Low sau fill
    max_oc = df[["Open","Close"]].max(axis=1)
    min_oc = df[["Open","Close"]].min(axis=1)
    df["High"] = np.maximum(df["High"], max_oc)
    df["Low"]  = np.minimum(df["Low"],  min_oc)

    return df

# Áp dụng cho tất cả symbol
cleaned_panels = []
for sym in symbols:
    sub = panel[sym].copy()             # cột cho 1 symbol
    sub = sub.reindex(close.index)      # đồng bộ index với close để dễ align
    sub_clean = clean_one_symbol(sub)
    cleaned_panels.append(sub_clean.rename(columns=lambda c: (sym, c)))

# Gộp lại MultiIndex columns
panel_clean = pd.concat(cleaned_panels, axis=1)
panel_clean.columns = pd.MultiIndex.from_tuples(panel_clean.columns)
panel_clean = panel_clean.sort_index(axis=1)    # sort theo (symbol, field)

# 3) Reindex về lịch ngày liên tục và cắt về đoạn chung
full_idx = pd.date_range(close.index.min(), close.index.max(), freq="D")
panel_clean = panel_clean.reindex(full_idx)

# Tìm đoạn chung mà tất cả symbol có Close không NaN
close_clean = panel_clean.xs("Close", axis=1, level=1)
first_valid = close_clean.apply(lambda s: s.first_valid_index())
last_valid  = close_clean.apply(lambda s: s.last_valid_index())
start_all   = max([d for d in first_valid if d is not None])
end_all     = min([d for d in last_valid  if d is not None])

panel_clean = panel_clean.loc[start_all:end_all]
close_clean = close_clean.loc[start_all:end_all]

print("Aligned (clean) range:", close_clean.index.min().date(), "→", close_clean.index.max().date())
print("Any NaN left in Close?", close_clean.isna().any().any())

# 4) Tính returns + winsorize nhẹ để khử outlier dữ liệu lỗi
rets = close_clean.pct_change().dropna()
logrets = np.log(close_clean).diff().dropna()

def winsorize_df(df, p=0.001):
    """Cắt đuôi p (0.1%) theo cột để hạn chế outlier do tick lỗi."""
    lo = df.quantile(p,   axis=0)
    hi = df.quantile(1-p, axis=0)
    return df.clip(lower=lo, upper=hi, axis=1)

rets_w  = winsorize_df(rets, p=0.001)
logr_w  = winsorize_df(logrets, p=0.001)

# 5) Lưu ra các file sạch để backtest
panel_clean_path = os.path.join(OUT_DIR, "um_1d_OHLCV_5y.CLEAN.parquet")
close_clean_path = os.path.join(OUT_DIR, "um_close_only_1d_aligned.CLEAN.parquet")
rets_path        = os.path.join(OUT_DIR, "um_close_returns_1d_aligned.CLEAN.parquet")
logrets_path     = os.path.join(OUT_DIR, "um_close_logreturns_1d_aligned.CLEAN.parquet")

panel_clean.to_parquet(panel_clean_path)
close_clean.to_parquet(close_clean_path)
rets_w.to_parquet(rets_path)
logr_w.to_parquet(logrets_path)

print("✅ Saved:")
print("  -", panel_clean_path)
print("  -", close_clean_path)
print("  -", rets_path)
print("  -", logrets_path)

# 6) Báo cáo QC nhanh
def brief(df, name):
    print(f"\n[{name}]")
    print("shape:", df.shape)
    print("range:", df.index.min().date(), "→", df.index.max().date())
    print("NaN % by column (top 5):")
    na = df.isna().mean().sort_values(ascending=False)
    print(na.head())

brief(close_clean, "CLOSE (CLEAN & ALIGNED)")
brief(rets_w,     "RETURNS (winsorized)")


Loaded panel: (1811, 35) | close: (1811, 7)
Date range (raw close): 2020-10-02 → 2025-09-30
Aligned (clean) range: 2020-10-02 → 2025-09-30
Any NaN left in Close? True
✅ Saved:
  - /content/drive/MyDrive/crypto_data/processed/um_1d_OHLCV_5y.CLEAN.parquet
  - /content/drive/MyDrive/crypto_data/processed/um_close_only_1d_aligned.CLEAN.parquet
  - /content/drive/MyDrive/crypto_data/processed/um_close_returns_1d_aligned.CLEAN.parquet
  - /content/drive/MyDrive/crypto_data/processed/um_close_logreturns_1d_aligned.CLEAN.parquet

[CLOSE (CLEAN & ALIGNED)]
shape: (1825, 7)
range: 2020-10-02 → 2025-09-30
NaN % by column (top 5):
BNBUSDT     0.007671
BTCUSDT     0.007671
DOGEUSDT    0.007671
ETHUSDT     0.007671
SOLUSDT     0.007671
dtype: float64

[RETURNS (winsorized)]
shape: (1824, 7)
range: 2020-10-03 → 2025-09-30
NaN % by column (top 5):
BNBUSDT     0.0
BTCUSDT     0.0
DOGEUSDT    0.0
ETHUSDT     0.0
SOLUSDT     0.0
dtype: float64


In [ ]:
# === Cell 6 — Make it perfectly filled for backtest ===
import os
import numpy as np
import pandas as pd

OUT_DIR = os.path.join(DUMP_ROOT, "processed")
src_close = os.path.join(OUT_DIR, "um_close_only_1d_aligned.CLEAN.parquet")
dst_close = os.path.join(OUT_DIR, "um_close_only_1d_filled.parquet")
dst_rets  = os.path.join(OUT_DIR, "um_close_returns_1d_filled.parquet")

# 1) Load close-only (đã aligned & clean)
close = pd.read_parquet(src_close).sort_index()
print("[BEFORE] NaN ratio by column (top 5):")
print(close.isna().mean().sort_values(ascending=False).head())

# 2) Đảm bảo index liên tục theo ngày (crypto 24/7)
full_idx = pd.date_range(close.index.min(), close.index.max(), freq="D")
close = close.reindex(full_idx)

# 3) Điền thiếu an toàn: ffill rồi bfill (điền lỗ rải rác; không đổi xu hướng dài hạn)
close_filled = close.ffill().bfill()

# 4) Sanity check: không còn NaN, giá > 0
if close_filled.isna().any().any():
    raise ValueError("Still has NaN after fill — please inspect columns with gaps.")
bad = (close_filled <= 0).any(axis=1)
if bad.any():
    # nếu có giá <=0 hiếm gặp do lỗi feed, thay bằng median hàng xóm (cực hiếm)
    med = close_filled.rolling(3, center=True).median()
    close_filled = close_filled.mask(close_filled <= 0, med).ffill().bfill()

# 5) Lưu bản filled cho backtest
close_filled.to_parquet(dst_close)
print("✅ Saved filled closes:", dst_close, "| shape:", close_filled.shape)

# 6) Tính returns lại trên bản filled (simple returns)
rets_filled = close_filled.pct_change().dropna()

# (tuỳ chọn) winsorize nhẹ để khử outlier dữ liệu lỗi còn sót
def winsorize_df(df, p=0.001):
    lo = df.quantile(p, axis=0)
    hi = df.quantile(1-p, axis=0)
    return df.clip(lower=lo, upper=hi, axis=1)

rets_filled = winsorize_df(rets_filled, p=0.001)
rets_filled.to_parquet(dst_rets)
print("✅ Saved filled returns:", dst_rets, "| shape:", rets_filled.shape)

# 7) Báo cáo nhanh
print("\n[AFTER] NaN ratio by column (should all be 0.0):")
print(close_filled.isna().mean().sort_values(ascending=False).head())
print("\nPreview tail (closes):")
display(close_filled.tail())
print("\nPreview tail (returns):")
display(rets_filled.tail())


[BEFORE] NaN ratio by column (top 5):
BNBUSDT     0.007671
BTCUSDT     0.007671
DOGEUSDT    0.007671
ETHUSDT     0.007671
SOLUSDT     0.007671
dtype: float64
✅ Saved filled closes: /content/drive/MyDrive/crypto_data/processed/um_close_only_1d_filled.parquet | shape: (1825, 7)
✅ Saved filled returns: /content/drive/MyDrive/crypto_data/processed/um_close_returns_1d_filled.parquet | shape: (1824, 7)

[AFTER] NaN ratio by column (should all be 0.0):
BNBUSDT     0.0
BTCUSDT     0.0
DOGEUSDT    0.0
ETHUSDT     0.0
SOLUSDT     0.0
dtype: float64

Preview tail (closes):


,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,SOLUSDT,TRXUSDT,XRPUSDT
2025-09-26,961.28,109588.5,0.23232,4030.17,205.01,0.33825,2.7823
2025-09-27,968.71,109577.3,0.23067,4017.01,203.32,0.33634,2.8032
2025-09-28,993.09,112119.6,0.23722,4139.55,210.70,0.33552,2.8662
2025-09-29,1030.43,114257.1,0.23469,4212.98,212.59,0.33672,2.8788
2025-09-30,1008.77,113988.8,0.23282,4142.75,208.60,0.33345,2.8454



Preview tail (returns):


,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,SOLUSDT,TRXUSDT,XRPUSDT
2025-09-26,0.015626,0.006004,0.042682,0.040554,0.064434,0.019501,0.014697
2025-09-27,0.007729,-0.000102,-0.007102,-0.003265,-0.008244,-0.005647,0.007512
2025-09-28,0.025167,0.023201,0.028396,0.030505,0.036297,-0.002438,0.022474
2025-09-29,0.037600,0.019064,-0.010665,0.017739,0.008970,0.003577,0.004396
2025-09-30,-0.021020,-0.002348,-0.007968,-0.016670,-0.018769,-0.009711,-0.011602


In [ ]:
# === Cell — Build 15 features (windows ≤ 60d), shifted để tránh look-ahead ===
import os
import numpy as np
import pandas as pd

OUT_DIR = os.path.join(DUMP_ROOT, "processed")

# ---- 0) Load các artifact đã có ----
panel_path      = os.path.join(OUT_DIR, "um_1d_OHLCV_5y.CLEAN.parquet")   # MultiIndex cols: (symbol, field)
close_filled_pq = os.path.join(OUT_DIR, "um_close_only_1d_filled.parquet") # full daily calendar, ffill/bfill
rets_filled_pq  = os.path.join(OUT_DIR, "um_close_returns_1d_filled.parquet")

assert os.path.exists(panel_path), f"Không thấy file: {panel_path}"
assert os.path.exists(close_filled_pq), f"Không thấy file: {close_filled_pq}"
assert os.path.exists(rets_filled_pq), f"Không thấy file: {rets_filled_pq}"

panel   = pd.read_parquet(panel_path)       # OHLCV sạch
closes  = pd.read_parquet(close_filled_pq)  # Close filled (DataFrame: columns=symbol)
rets    = pd.read_parquet(rets_filled_pq)   # daily returns (simple), winsorized nhẹ

# Đồng bộ index và danh sách symbol
symbols = sorted({c[0] for c in panel.columns})
closes  = closes[symbols]
rets    = rets[symbols]
panel   = panel.reindex(closes.index)  # đảm bảo chung index full daily

# Tiện ích: rolling corr theo cột
def rolling_corr(s1: pd.Series, s2: pd.Series, win: int):
    return s1.rolling(win).corr(s2)

# ---- 1) Lấy OHLCV từng mã (đã sạch) ----
opens   = panel.xs("Open",   axis=1, level=1)[symbols]
highs   = panel.xs("High",   axis=1, level=1)[symbols]
lows    = panel.xs("Low",    axis=1, level=1)[symbols]
volumes = panel.xs("volume", axis=1, level=1)[symbols]
# (Close dùng bản filled để liền mạch)
closes  = closes[symbols]

# ---- 2) Nhóm A. Momentum & Trend ----
# 1) Daily Return (r1): đã có ở rets
feat_r1 = rets.copy()

# 2) Momentum 20d & 3) Momentum 60d — dùng % thay đổi so với giá 20/60 phiên trước
feat_mom20 = closes.pct_change(20)
feat_mom60 = closes.pct_change(60)

# 4) Return mean 7 ngày gần nhất
feat_retmean7 = rets.rolling(7).mean()

# 5) SMA60 filter — 1 nếu Close > SMA60, else 0 (để rule "chỉ long khi >0")
sma60 = closes.rolling(60).mean()
feat_sma60_filter = (closes > sma60).astype(int)

# ---- 3) Nhóm B. Volatility & Risk ----
# 6) Rolling Volatility σ20: độ lệch chuẩn return 20d
feat_vol20 = rets.rolling(20).std()

# 7) ATR14 (Average True Range, absolute)
# True Range = max(High-Low, |High - prevClose|, |Low - prevClose|)
prev_close = closes.shift(1)
tr = pd.concat([
    (highs - lows),
    (highs - prev_close).abs(),
    (lows  - prev_close).abs()
], axis=1).max(axis=1)  # max across the 3 stacked pieces row-wise
# Vì concat theo cột, cần reshaping để từng symbol: làm TR cho từng symbol riêng:
tr_sym = pd.DataFrame(index=closes.index, columns=symbols, dtype=float)
for sym in symbols:
    a = (highs[sym] - lows[sym])
    b = (highs[sym] - prev_close[sym]).abs()
    c = (lows[sym]  - prev_close[sym]).abs()
    tr_sym[sym] = pd.concat([a, b, c], axis=1).max(axis=1)

feat_atr14 = tr_sym.rolling(14).mean()
# (Nếu muốn chuẩn hoá: feat_atr14_norm = (feat_atr14 / closes))

# 8) Volatility Ratio (σ10 / σ60)
sigma10 = rets.rolling(10).std()
sigma60 = rets.rolling(60).std()
feat_vol_ratio_10_60 = sigma10 / sigma60

# 9) Max Drawdown (local 60d): min drawdown trong cửa sổ 60d (giá/đỉnh_cục_bộ - 1)
roll_max60 = closes.rolling(60).max()
drawdown = closes / roll_max60 - 1.0
feat_mdd60 = drawdown.rolling(60).min()  # giá trị âm (tệ nhất trong 60d gần nhất)

# ---- 4) Nhóm C. Volume & Market Confirmation ----
# 10) Volume Ratio 20d = Volume / MA20(Volume)
vol_ma20 = volumes.rolling(20).mean()
feat_volratio20 = volumes / vol_ma20

# 11) OBV Change 20d
#   OBV: tích luỹ +vol khi Close tăng, -vol khi Close giảm, 0 nếu bằng
r_sign = np.sign(rets.fillna(0.0))
obv = (r_sign * volumes).cumsum()
feat_obv_change20 = obv.pct_change(20)  # thay đổi tương đối 20 phiên

# 12) Price–Volume correlation 30d: corr(ΔP, ΔVol), dùng return & pct_change(volume)
dvol = volumes.pct_change().replace([np.inf, -np.inf], np.nan)
feat_pv_corr30 = pd.DataFrame(index=closes.index, columns=symbols, dtype=float)
for sym in symbols:
    feat_pv_corr30[sym] = rets[sym].rolling(30).corr(dvol[sym])

# ---- 5) Nhóm D. Relative Strength & Cross-asset ----
# 13) Relative Momentum (coin – mean basket) trên 60d
mom60_mean = feat_mom60.mean(axis=1)
feat_rel_mom60 = feat_mom60.subtract(mom60_mean, axis=0)

# 14) Beta vs BTC (corr60) — theo mô tả của bạn là correlation 60d với BTC
btc = "BTCUSDT"
assert btc in symbols, "Danh mục không có BTCUSDT — cần BTC để tính corr60."
feat_corr60_btc = pd.DataFrame(index=closes.index, columns=symbols, dtype=float)
for sym in symbols:
    feat_corr60_btc[sym] = rets[sym].rolling(60).corr(rets[btc])

# (Nếu muốn beta thực sự: beta60 = corr60 * (σ_coin60 / σ_btc60))

# 15) Market breadth (share of coins > SMA60) — tính chung rồi broadcast cho từng coin
above_sma = (closes > sma60).astype(int)
breadth = above_sma.sum(axis=1) / above_sma.shape[1]
feat_breadth = pd.DataFrame(
    np.repeat(breadth.values.reshape(-1,1), len(symbols), axis=1),
    index=closes.index, columns=symbols
)

# ---- 6) Gom các feature vào một panel MultiIndex (symbol, feature) ----
features_dict = {
    "ret1":               feat_r1,
    "mom20":              feat_mom20,
    "mom60":              feat_mom60,
    "retmean7":           feat_retmean7,
    "sma60_filter":       feat_sma60_filter,

    "vol20":              feat_vol20,
    "atr14":              feat_atr14,
    "vol_ratio_10_60":    feat_vol_ratio_10_60,
    "mdd60":              feat_mdd60,

    "vol_ratio20":        feat_volratio20,
    "obv_change20":       feat_obv_change20,
    "pv_corr30":          feat_pv_corr30,

    "rel_mom60":          feat_rel_mom60,
    "corr60_btc":         feat_corr60_btc,
    "breadth_sma60":      feat_breadth,
}

# Nối theo trục cột với MultiIndex (symbol, feature_name)
stacked = []
for feat_name, df in features_dict.items():
    df2 = df.copy()
    # Bảo đảm cột theo đúng thứ tự symbol
    df2 = df2[symbols]
    # Đặt MultiIndex cột: (symbol, feature)
    df2.columns = pd.MultiIndex.from_product([symbols, [feat_name]])
    stacked.append(df2)

features_panel = pd.concat(stacked, axis=1).sort_index(axis=1)

# ---- 7) Tránh look-ahead: shift toàn bộ feature 1 ngày ----
# Quy ước: dùng feature tại ngày t-1 để đưa ra quyết định cho ngày t (rebalance mỗi 5 ngày cũng OK)
features_panel = features_panel.shift(1)

# ---- 8) Cắt bỏ phần đầu chưa đủ dữ liệu (60d) ----
min_start = closes.index[60]  # đủ cho mọi cửa sổ ≤60
features_panel = features_panel.loc[min_start:]

# ---- 9) Lưu & báo cáo ----
dst_parquet = os.path.join(OUT_DIR, "um_features_15.parquet")
dst_csv     = os.path.join(OUT_DIR, "um_features_15.csv")
features_panel.to_parquet(dst_parquet)
# Flatten cột để CSV gọn: symbol__feature
flat = features_panel.copy()
flat.columns = [f"{c[0]}__{c[1]}" for c in flat.columns]
flat.to_csv(dst_csv, encoding="utf-8")

print("✅ Saved 15-feature panel:")
print("  -", dst_parquet)
print("  -", dst_csv)
print("Symbols:", symbols)
print("Feature starts:", features_panel.index.min().date(), "→", features_panel.index.max().date())
print("Columns example:", features_panel.columns[:8])


✅ Saved 15-feature panel:
  - /content/drive/MyDrive/crypto_data/processed/um_features_15.parquet
  - /content/drive/MyDrive/crypto_data/processed/um_features_15.csv
Symbols: ['BNBUSDT', 'BTCUSDT', 'DOGEUSDT', 'ETHUSDT', 'SOLUSDT', 'TRXUSDT', 'XRPUSDT']
Feature starts: 2020-12-01 → 2025-09-30
Columns example: MultiIndex([('BNBUSDT',         'atr14'),
            ('BNBUSDT', 'breadth_sma60'),
            ('BNBUSDT',    'corr60_btc'),
            ('BNBUSDT',         'mdd60'),
            ('BNBUSDT',         'mom20'),
            ('BNBUSDT',         'mom60'),
            ('BNBUSDT',  'obv_change20'),
            ('BNBUSDT',     'pv_corr30')],
           )


In [ ]:
# === Cell — Beautiful dashboard for 15 features over full 5 years (Plotly dark) ===
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

OUT_DIR = os.path.join(DUMP_ROOT, "processed")
viz_dir = os.path.join(OUT_DIR, "visualizations")
os.makedirs(viz_dir, exist_ok=True)

# ---- Load data ----
feat_path  = os.path.join(OUT_DIR, "um_features_15.parquet")
close_path = os.path.join(OUT_DIR, "um_close_only_1d_filled.parquet")

assert os.path.exists(feat_path),  f"Missing: {feat_path}"
assert os.path.exists(close_path), f"Missing: {close_path}"

F = pd.read_parquet(feat_path)    # MultiIndex columns: (symbol, feature)
C = pd.read_parquet(close_path)   # columns = symbols

symbols = sorted({c[0] for c in F.columns})
date_idx = F.index

def feat(sym, name):
    return F[(sym, name)].rename(name)

# breadth giống nhau giữa các coin (đã broadcast)
BREADTH = feat(symbols[0], "breadth_sma60")

# ---- Mini stats helper (12M gần nhất) ----
def minime_stats(s):
    s = s.dropna()
    if s.empty: return (np.nan, np.nan, np.nan)
    return (float(s.min()), float(s.mean()), float(s.max()))

def last_n_months(series, months=12):
    cutoff = series.index.max() - pd.DateOffset(months=months)
    return series.loc[series.index >= cutoff]

# ---- Build dashboard for one symbol ----
def make_dashboard(sym: str):
    title = f"{sym} — Feature Overview (Momentum, Vol/Risk, Volume, Relative; 5y)"
    fig = make_subplots(
        rows=6, cols=1, shared_xaxes=True, vertical_spacing=0.03,
        specs=[
            [{"secondary_y": True}],  # 1) Price (+SMA60 + filter band)
            [{"secondary_y": False}], # 2) Momentum
            [{"secondary_y": True}],  # 3) Vol & Risk
            [{"secondary_y": False}], # 4) Volume & Confirmation
            [{"secondary_y": False}], # 5) Relative/Cross
            [{"secondary_y": False}], # 6) Minime stats (as text)
        ],
        row_heights=[0.26, 0.18, 0.19, 0.16, 0.15, 0.06]
    )

    # ---------- (1) PRICE ----------
    close  = C[sym]
    sma60  = close.rolling(60).mean()
    flt    = feat(sym, "sma60_filter").fillna(0)
    shade  = flt.replace({0: np.nan, 1: close.max()*0.02})

    fig.add_trace(go.Scatter(x=date_idx, y=close, name="Close", mode="lines"), row=1, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=date_idx, y=sma60, name="SMA60", mode="lines", line=dict(dash="dot")),
                  row=1, col=1, secondary_y=False)
    # thin band to show filter ON
    fig.add_trace(go.Scatter(x=date_idx, y=shade, name="Only-Long (SMA60>0)", mode="lines",
                             fill="tozeroy", opacity=0.25, hoverinfo="skip"),
                  row=1, col=1, secondary_y=True)
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Filter", row=1, col=1, secondary_y=True, showgrid=False)

    # ---------- (2) MOMENTUM ----------
    r1       = feat(sym, "ret1")
    mom20    = feat(sym, "mom20")
    mom60    = feat(sym, "mom60")
    retmean7 = feat(sym, "retmean7")

    for s, nm in [(r1, "r₁ (daily)"), (mom20, "mom20"), (mom60, "mom60"), (retmean7, "retmean7")]:
        fig.add_trace(go.Scatter(x=date_idx, y=s, name=nm, mode="lines"), row=2, col=1)

    fig.add_hline(y=0, line_width=1, line_dash="dot", row=2, col=1)
    fig.update_yaxes(title_text="Momentum & Returns", row=2, col=1)

    # ---------- (3) VOLATILITY & RISK ----------
    vol20    = feat(sym, "vol20")
    atr14    = feat(sym, "atr14")
    vr_10_60 = feat(sym, "vol_ratio_10_60")
    mdd60    = feat(sym, "mdd60")

    fig.add_trace(go.Scatter(x=date_idx, y=vol20, name="σ20", mode="lines"),
                  row=3, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=date_idx, y=atr14, name="ATR14 (abs)", mode="lines"),
                  row=3, col=1, secondary_y=True)
    fig.add_trace(go.Scatter(x=date_idx, y=vr_10_60, name="σ10/σ60", mode="lines"),
                  row=3, col=1, secondary_y=True)
    fig.add_trace(go.Scatter(x=date_idx, y=mdd60, name="MDD60 (min)", mode="lines"),
                  row=3, col=1, secondary_y=False)

    # Baselines
    fig.add_hline(y=0, line_width=1, line_dash="dot", row=3, col=1, secondary_y=False)  # for MDD
    fig.add_hline(y=1, line_width=1, line_dash="dot", row=3, col=1, secondary_y=True)   # for ratio

    fig.update_yaxes(title_text="σ20 / MDD60", row=3, col=1, secondary_y=False)
    fig.update_yaxes(title_text="ATR / σ-ratio", row=3, col=1, secondary_y=True, showgrid=False)

    # ---------- (4) VOLUME & CONFIRMATION ----------
    volratio20 = feat(sym, "vol_ratio20")
    obvchg20   = feat(sym, "obv_change20")
    pvcorr30   = feat(sym, "pv_corr30")

    fig.add_trace(go.Scatter(x=date_idx, y=volratio20, name="Volume Ratio 20d", mode="lines"), row=4, col=1)
    fig.add_trace(go.Scatter(x=date_idx, y=obvchg20,   name="OBV Change 20d",   mode="lines"), row=4, col=1)
    fig.add_trace(go.Scatter(x=date_idx, y=pvcorr30,   name="Corr(ΔP,ΔVol) 30d",mode="lines"), row=4, col=1)

    fig.add_hline(y=1, line_width=1, line_dash="dot", row=4, col=1)  # volume baseline
    fig.add_hline(y=0, line_width=1, line_dash="dot", row=4, col=1)  # corr baseline
    fig.update_yaxes(title_text="Volume / Confirmation", row=4, col=1)

    # ---------- (5) RELATIVE / CROSS-ASSET ----------
    rel_mom60  = feat(sym, "rel_mom60")
    corr60btc  = feat(sym, "corr60_btc")
    breadth    = BREADTH

    fig.add_trace(go.Scatter(x=date_idx, y=rel_mom60, name="Relative Momentum 60d", mode="lines"), row=5, col=1)
    fig.add_trace(go.Scatter(x=date_idx, y=corr60btc, name="Corr60 with BTC",      mode="lines"), row=5, col=1)
    fig.add_trace(go.Scatter(x=date_idx, y=breadth,   name="Market Breadth (>%SMA60)", mode="lines"),
                  row=5, col=1)

    fig.add_hline(y=0, line_width=1, line_dash="dot", row=5, col=1)  # baseline rel_mom, corr
    fig.update_yaxes(title_text="Relative / Cross", row=5, col=1)

    # ---------- (6) MINI-STATS ----------
    # 12 tháng gần nhất để đọc nhanh regime
    r = last_n_months(r1, 12)
    v = last_n_months(vol20, 12)
    vr = last_n_months(vr_10_60, 12)
    pv = last_n_months(pvcorr30, 12)
    br = last_n_months(breadth, 12)

    txt = []
    txt.append(f"r₁ (12M)  min/mean/max: {minime_stats(r)}")
    txt.append(f"σ20 (12M) min/mean/max: {minime_stats(v)}")
    txt.append(f"σ10/σ60 (12M) around 1 → {minime_stats(vr)}")
    txt.append(f"PV corr (12M)          : {minime_stats(pv)}")
    txt.append(f"Breadth (12M) 0→1      : {minime_stats(br)}")

    fig.add_trace(go.Scatter(
        x=[date_idx.min()], y=[0], mode="text",
        text="<br>".join(txt), textposition="top left", showlegend=False
    ), row=6, col=1)
    fig.update_yaxes(visible=False, row=6, col=1)

    # ---- Layout ----
    fig.update_layout(
        template="plotly_dark",
        title=title,
        height=1250,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(l=60, r=25, t=70, b=40),
        hovermode="x unified"
    )
    fig.update_xaxes(showspikes=True)
    fig.update_yaxes(showspikes=True)
    return fig

# ---- Build first and show ----
main_fig = make_dashboard(symbols[0])
main_fig.show()

# ---- Export HTML for every symbol (shareable) ----
for sym in symbols:
    fig = make_dashboard(sym)
    out_html = os.path.join(viz_dir, f"features15_{sym}.html")
    fig.write_html(out_html, include_plotlyjs="cdn")
    print("✅ Saved:", out_html)

# ---- Optional: overall heatmap for corr60_btc & rel_mom60 (multi-coin quick view) ----
corr_mat = pd.DataFrame({sym: feat(sym, "corr60_btc") for sym in symbols})
relm_mat = pd.DataFrame({sym: feat(sym, "rel_mom60")  for sym in symbols})

for name, mat in [("corr60_btc", corr_mat), ("rel_mom60", relm_mat)]:
    m = mat.dropna(how="all")
    fig_hm = go.Figure(data=go.Heatmap(z=m[symbols].T.values, x=m.index, y=symbols, coloraxis="coloraxis"))
    fig_hm.update_layout(
        template="plotly_dark",
        title=f"{name} — heatmap (rows=coin, cols=time)",
        coloraxis={"colorscale": "RdBu", "cmin": -1.0, "cmax": 1.0},
        height=400 + 18*len(symbols),
        margin=dict(l=100, r=20, t=50, b=40)
    )
    out_html = os.path.join(viz_dir, f"{name}_heatmap.html")
    fig_hm.write_html(out_html, include_plotlyjs="cdn")
    print("✅ Saved:", out_html)


✅ Saved: /content/drive/MyDrive/crypto_data/processed/visualizations/features15_BNBUSDT.html
✅ Saved: /content/drive/MyDrive/crypto_data/processed/visualizations/features15_BTCUSDT.html
✅ Saved: /content/drive/MyDrive/crypto_data/processed/visualizations/features15_DOGEUSDT.html
✅ Saved: /content/drive/MyDrive/crypto_data/processed/visualizations/features15_ETHUSDT.html
✅ Saved: /content/drive/MyDrive/crypto_data/processed/visualizations/features15_SOLUSDT.html
✅ Saved: /content/drive/MyDrive/crypto_data/processed/visualizations/features15_TRXUSDT.html
✅ Saved: /content/drive/MyDrive/crypto_data/processed/visualizations/features15_XRPUSDT.html
✅ Saved: /content/drive/MyDrive/crypto_data/processed/visualizations/corr60_btc_heatmap.html
✅ Saved: /content/drive/MyDrive/crypto_data/processed/visualizations/rel_mom60_heatmap.html


In [ ]:
# === Cell — Rule-based backtest (anti-lookahead), 4d rebalance, TP/SL, equal-weight
#            + Audits + Visualizations (Drawdown, Rolling Sharpe, Picks, Allocation) ===
import os, numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

OUT_DIR = os.path.join(DUMP_ROOT, "processed")
close_path = os.path.join(OUT_DIR, "um_close_only_1d_filled.parquet")
feat_path  = os.path.join(OUT_DIR, "um_features_15.parquet")
assert os.path.exists(close_path) and os.path.exists(feat_path), "Thiếu file dữ liệu đã chuẩn bị."

# ----------------------- Hyperparameters (dễ chỉnh) -----------------------
RSI_MIN      = 55
VOL20_MAX    = 0.10
MOM60_MIN    = 0.12
RET1_MIN     = -0.02
BREADTH_MIN  = 0.60
TP_PCT       = 0.15
SL_PCT       = -0.03
FEE_PCT      = 0.0012
REBAL_DAYS   = 4
TOP_K        = 5
ROLL_WIN     = 90

# ----------------------- Load data -----------------------
closes   = pd.read_parquet(close_path).sort_index()
features = pd.read_parquet(feat_path).sort_index()
symbols  = sorted({c[0] for c in features.columns})
dates    = closes.index

def feat(sym, name): return features[(sym, name)]

# RSI14 từ close (sẽ shift(1) để dùng T-1)
def calc_rsi(close, window=14):
    d = close.diff()
    up, dn = d.clip(lower=0), -d.clip(upper=0)
    rs = up.rolling(window).mean() / dn.rolling(window).mean().replace(0, np.nan)
    return 100 - 100/(1+rs)

RSI14 = pd.DataFrame({s: calc_rsi(closes[s]) for s in symbols}).shift(1)

# ----------------------- AUDIT 1: mốc thời gian -----------------------
print("[AUDIT] Close starts:", closes.index.min().date())
print("[AUDIT] Features starts:", features.index.min().date())
probe = [(symbols[0],'mom60'), (symbols[0],'vol20'), (symbols[0],'ret1'),
         (symbols[0],'sma60_filter'), (symbols[0],'breadth_sma60')]
for col in probe:
    s = features[col].dropna()
    if len(s): print(f"[AUDIT] first_valid {col}: {s.index[0].date()}")

# ----------------------- Build signals (raw at day t) -----------------------
sig_raw = pd.DataFrame(0, index=dates, columns=symbols)
for s in symbols:
    cond = (
        (feat(s,"mom60") > MOM60_MIN) &
        (RSI14[s]       > RSI_MIN) &
        (feat(s,"vol20") < VOL20_MAX) &
        (feat(s,"ret1")  > RET1_MIN) &
        (feat(s,"sma60_filter") > 0) &
        (feat(s,"breadth_sma60") > BREADTH_MIN)
    )
    sig_raw[s] = cond.astype(int)

# dùng tín hiệu T-1 cho quyết định ngày t
sig = sig_raw.shift(1).fillna(0).astype(int)

# ----------------------- AUDIT 2: corr(signal_t, r_t) vs corr(signal_T-1, r_t) -----------------------
flat_r  = closes.pct_change().stack().dropna()
sig_t   = sig_raw.stack().reindex(flat_r.index).fillna(0)   # tín hiệu cùng ngày
sig_tm1 = sig.stack().reindex(flat_r.index).fillna(0)       # tín hiệu lùi 1 ngày
print(f"[AUDIT] Corr(signal_t, ret_t)={sig_t.corr(flat_r):.4f} | Corr(signal_T-1, ret_t)={sig_tm1.corr(flat_r):.4f}")

# ----------------------- Backtest (T-1 signals; TP/SL chỉ khi đã cầm) -----------------------
rets = closes.pct_change().fillna(0.0)
equity   = pd.Series(index=dates, dtype=float)
weights  = pd.DataFrame(0.0, index=dates, columns=symbols)
picks_ct = pd.Series(0, index=dates, dtype=int)

value = 1.0
prev_w = pd.Series(0.0, index=symbols)

for i, d in enumerate(dates):
    if i == 0:
        equity[d] = value
        continue

    today_w = prev_w.copy()

    # Rebalance: dùng signal của ngày T-1
    if i % REBAL_DAYS == 0:
        d_sig = dates[i-1]
        candidates = [s for s in symbols if sig.loc[d_sig, s] == 1]
        if len(candidates) > TOP_K:
            moms_tm1 = {s: features.loc[d_sig, (s,"mom60")] for s in candidates}
            candidates = sorted(candidates, key=lambda x: moms_tm1[x], reverse=True)[:TOP_K]

        today_w[:] = 0.0
        if len(candidates) > 0:
            today_w[candidates] = 1.0/len(candidates)

        cost = FEE_PCT * (today_w != 0).sum()
    else:
        cost = 0.0

    # TP/SL: chỉ áp dụng nếu vị thế đã tồn tại từ hôm qua (tránh check ngay ngày vào)
    for s in symbols:
        if today_w[s] != 0 and prev_w[s] != 0:
            r = rets.loc[d, s]
            if r >= TP_PCT or r <= SL_PCT:
                today_w[s] = 0.0

    port_ret = float((today_w * rets.loc[d]).sum()) - cost

    value *= (1 + port_ret)
    equity[d]   = value
    weights.loc[d] = today_w
    picks_ct[d] = int((today_w != 0).sum())
    prev_w = today_w

# ----------------------- AUDIT 3: ngày action vs ngày signal -----------------------
print("[AUDIT] action day (t)  |  signal day (t-1)")
for d_act in dates[::REBAL_DAYS][:8]:
    if d_act == dates[0]: continue
    d_sig = dates[dates.get_loc(d_act)-1]
    print(" ", d_act.date(), "|", d_sig.date())

# ----------------------- Metrics -----------------------
def max_drawdown(e):
    peak = e.cummax(); dd = e/peak - 1.0
    return dd.min(), dd

years = (dates[-1] - dates[0]).days/365.0
daily_ret = equity.pct_change().dropna()
ann_ret = (equity.iloc[-1]/equity.iloc[0])**(1/years) - 1
ann_vol = daily_ret.std()*np.sqrt(365)
sharpe  = ann_ret/ann_vol if ann_vol>0 else np.nan
mdd, dd_series = max_drawdown(equity)
hit_ratio = (daily_ret>0).mean()*100
avg_daily_cost = FEE_PCT/REBAL_DAYS*100

print(f"\nStart: {dates[0].date()}  |  End: {dates[-1].date()}  |  Days: {len(dates)}")
print(f"Annual Return (CAGR): {ann_ret*100:.2f}%")
print(f"Annual Volatility   : {ann_vol*100:.2f}%")
print(f"Sharpe (rf=0)       : {sharpe:.2f}")
print(f"Max Drawdown (MDD)  : {mdd*100:.2f}%")
print(f"Hit Ratio (daily>0) : {hit_ratio:.2f}%")
print(f"Avg picks per rebal : {picks_ct[dates[::REBAL_DAYS]].mean():.2f} coins")
print(f"Avg daily cost      : {avg_daily_cost:.4f}%")

# ----------------------- Visualizations -----------------------
# (A) Equity + Drawdown
figA = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, specs=[[{}],[{}]])
figA.add_trace(go.Scatter(x=equity.index, y=equity.values, name="Equity", mode="lines"), row=1,col=1)
figA.update_yaxes(title_text="Equity", row=1,col=1)
figA.add_trace(go.Scatter(x=dd_series.index, y=dd_series.values, name="Drawdown", mode="lines", fill="tozeroy"), row=2,col=1)
figA.update_yaxes(title_text="Drawdown", tickformat=".0%", row=2,col=1)
figA.update_layout(template="plotly_dark", title="Portfolio Equity & Underwater Drawdown (anti-lookahead)",
                   height=650, margin=dict(l=60,r=20,t=60,b=40), hovermode="x unified")
figA.show()

# (B) Rolling Sharpe
roll = daily_ret.rolling(ROLL_WIN)
roll_sharpe = (roll.mean()/roll.std())*np.sqrt(365)
figB = go.Figure()
figB.add_trace(go.Scatter(x=roll_sharpe.index, y=roll_sharpe.values, mode="lines", name=f"Rolling Sharpe ({ROLL_WIN}d)"))
figB.add_hline(y=1.0, line_dash="dot", annotation_text="Sharpe=1")
figB.update_layout(template="plotly_dark", title=f"Rolling Sharpe ({ROLL_WIN}-day window)",
                   height=350, margin=dict(l=60,r=20,t=50,b=40))
figB.show()

# (C) Picks per Rebalance
reb_dates = dates[::REBAL_DAYS]
figC = go.Figure()
figC.add_trace(go.Bar(x=reb_dates, y=picks_ct.loc[reb_dates], name="#picks",
                      width=0.8*24*3600*1000*REBAL_DAYS))
figC.update_layout(template="plotly_dark", title="Number of Coins Picked at Each Rebalance",
                   xaxis_title="Date", yaxis_title="#Coins", height=300, margin=dict(l=60,r=20,t=50,b=40))
figC.show()

# (D) Allocation heatmap
wm = weights.copy()
figD = go.Figure(data=go.Heatmap(z=wm[symbols].T.values, x=wm.index, y=symbols, coloraxis="coloraxis"))
figD.update_layout(template="plotly_dark", title="Allocation Heatmap (daily weights)",
                   coloraxis={"colorscale":"Viridis","cmin":0.0,"cmax":1.0},
                   height=400+18*len(symbols), margin=dict(l=100,r=20,t=50,b=40))
figD.show()


[AUDIT] Close starts: 2020-10-02
[AUDIT] Features starts: 2020-12-01
[AUDIT] first_valid ('BNBUSDT', 'mom60'): 2020-12-02
[AUDIT] first_valid ('BNBUSDT', 'vol20'): 2020-12-01
[AUDIT] first_valid ('BNBUSDT', 'ret1'): 2020-12-01
[AUDIT] first_valid ('BNBUSDT', 'sma60_filter'): 2020-12-01
[AUDIT] first_valid ('BNBUSDT', 'breadth_sma60'): 2020-12-01
[AUDIT] Corr(signal_t, ret_t)=0.0195 | Corr(signal_T-1, ret_t)=0.0243
[AUDIT] action day (t)  |  signal day (t-1)
  2020-10-06 | 2020-10-05
  2020-10-10 | 2020-10-09
  2020-10-14 | 2020-10-13
  2020-10-18 | 2020-10-17
  2020-10-22 | 2020-10-21
  2020-10-26 | 2020-10-25
  2020-10-30 | 2020-10-29

Start: 2020-10-02  |  End: 2025-09-30  |  Days: 1825
Annual Return (CAGR): 175.96%
Annual Volatility   : 32.00%
Sharpe (rf=0)       : 5.50
Max Drawdown (MDD)  : -14.32%
Hit Ratio (daily>0) : 17.00%
Avg picks per rebal : 0.91 coins
Avg daily cost      : 0.0300%
